In [2]:
# ============================================================
# Bronze: Carga de archivos Excel de generación EERSA 2021
# Lista los 12 .xlsx desde la zona raw del Lakehouse
# ============================================================

from notebookutils import mssparkutils

RAW_PATH = "Files/raw_eersa_generacion_2021"

archivos = mssparkutils.fs.ls(RAW_PATH)
xlsx_files = sorted([f for f in archivos if f.name.endswith(".xlsx")], key=lambda f: f.name)

print(f"Archivos encontrados: {len(xlsx_files)}")
for f in xlsx_files:
    print(f"  - {f.name} ({f.size} bytes)")

StatementMeta(, 08e49674-7cfa-41e4-a105-cc98de505606, 4, Finished, Available, Finished, False)

Archivos encontrados: 12
  - 01 RESUMEN ENERO 2021.xlsx (22141 bytes)
  - 02 RESUMEN FEBRERO 2021.xlsx (21732 bytes)
  - 03 RESUMEN MARZO 2021.xlsx (21876 bytes)
  - 04 RESUMEN ABRIL 2021.xlsx (21509 bytes)
  - 05 RESUMEN MAYO 2021.xlsx (21522 bytes)
  - 06 RESUMEN JUNIO 2021.xlsx (21451 bytes)
  - 07 RESUMEN JULIO 2021.xlsx (21561 bytes)
  - 08  RESUMEN AGOSTO 2021.xlsx (21369 bytes)
  - 09 RESUMEN SEPTIEMBRE 2021.xlsx (21692 bytes)
  - 10 RESUMEN OCTUBRE 2021.xlsx (22637 bytes)
  - 11 RESUMEN NOVIEMBRE 2021.xlsx (22164 bytes)
  - 12 RESUMEN DICIEMBRE 2021.xlsx (25229 bytes)


In [3]:
# ============================================================
# Test: leer un archivo de prueba con pandas + openpyxl
# ============================================================

import pandas as pd

# Ruta local del Lakehouse montado (no la ABFSS)
LOCAL_RAW = "/lakehouse/default/Files/raw_eersa_generacion_2021"

# Probamos con enero
test_file = f"{LOCAL_RAW}/01 RESUMEN ENERO 2021.xlsx"

# Leer la hoja RESUMEN, headers en filas 9-10 (índice 8-9)
df = pd.read_excel(
    test_file,
    sheet_name="RESUMEN",
    header=[8, 9],   # headers en dos niveles
    engine="openpyxl"
)

print(f"Shape: {df.shape}")
print(f"\nPrimeras 5 filas:")
df.head()

StatementMeta(, 08e49674-7cfa-41e4-a105-cc98de505606, 5, Finished, Available, Finished, False)

Shape: (57, 23)

Primeras 5 filas:


Unnamed: 0_level_0                DIA       ALAO                        \
  Unnamed: 0_level_1 Unnamed: 1_level_1         KW E.Bruta kWh C.Int kWh   
0                NaN                  1  10044.586  238778.318     100.0   
1                NaN                  2  10059.704  241562.288     150.0   
2                NaN                  3  10074.495  242208.644     200.0   
3                NaN                  4   9956.266  239610.407     200.0   
4                NaN                  5  10053.612  241460.754     200.0   

              RIO BLANCO                                   ...    C.NIZAG  \
   E.Neta kWh         KW E.Bruta kWh C.Int kWh E.Neta kWh  ... E.Neta Kwh   
0  238678.318   1192.340   37472.408      90.0  37382.408  ...   12165.94   
1  241412.288   1142.892   38242.812      96.0  38146.812  ...   12106.90   
2  242008.644   1191.775   38241.710      92.0  38149.710  ...   12322.08   
3  239410.407   1191.503   38114.497      88.0  38026.497  ...   12734.62   
4  241260.754   1191.422   38125.878      84.0  38041.878  ...   13355.70   

         S.N.I.                                 TOTAL EERSA               \
             KW         KWh KWh.1 KWh.2 KWh.3            KW         KW-H   
0  51291.782667  831027.305   NaN   NaN   NaN  63038.708667  1119502.031   
1  53184.658000  857255.419   NaN   NaN   NaN     64898.254  1149224.519   
2  47568.992667  758864.961   NaN    dm   NaN  59345.262667  1051695.315   
3  58860.907000  860281.469   NaN   NaN   NaN     70569.676  1150798.373   
4  51707.623333  872737.631   NaN   NaN   NaN  63504.657333  1165740.263   

  ENERGIA ENTREGADA AL MEM (kWh) ENERGIA RECIBIDA DEL MEM (kWH)  
             Unnamed: 21_level_1            Unnamed: 22_level_1  
0                     276060.726                    1107088.031  
1                     279559.100                    1136814.519  
2                     280158.354                    1039023.315  
3                     277436.904                    1137718.373  
4                     279302.632                    1152040.263  

[5 rows x 23 columns]

In [6]:
# ============================================================
# Bronze: procesar los 12 xlsx y escribir tabla Delta
# Versión robusta al split de planta__metrica
# ============================================================

import uuid
from datetime import datetime

LOCAL_RAW = "/lakehouse/default/Files/raw_eersa_generacion_2021"
BATCH_ID = str(uuid.uuid4())
INGESTED_AT = datetime.utcnow()

MESES = {
    "ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4,
    "MAYO": 5, "JUNIO": 6, "JULIO": 7, "AGOSTO": 8,
    "SEPTIEMBRE": 9, "OCTUBRE": 10, "NOVIEMBRE": 11, "DICIEMBRE": 12
}

def extraer_mes_anio(filename):
    partes = filename.upper().replace(".XLSX", "").split()
    mes_num = next((MESES[p] for p in partes if p in MESES), None)
    anio = next((int(p) for p in partes if p.isdigit() and len(p) == 4), None)
    return mes_num, anio

todos_df = []

for f in xlsx_files:
    nombre = f.name
    mes, anio = extraer_mes_anio(nombre)
    ruta = f"{LOCAL_RAW}/{nombre}"
    
    df = pd.read_excel(ruta, sheet_name="RESUMEN", header=[8, 9], engine="openpyxl")
    
    # Aplanar columnas multinivel: solo conservar las que tengan planta válida en level_0
    nuevas_cols = []
    for a, b in df.columns:
        a_str = str(a).strip()
        b_str = str(b).strip()
        if "Unnamed" in a_str:
            # Es la columna día u otra columna sin planta arriba
            nuevas_cols.append(b_str)
        else:
            nuevas_cols.append(f"{a_str}__{b_str}")
    df.columns = nuevas_cols
    
    # Detectar columna del día (es la que tiene valores numéricos 1-31)
    col_dia = None
    for c in df.columns:
        if c.upper() in ("DIA", "DÍA"):
            col_dia = c
            break
    if col_dia is None:
        # Fallback: la segunda columna suele ser el día
        col_dia = df.columns[1]
    
    df = df.rename(columns={col_dia: "dia"})
    df = df[pd.to_numeric(df["dia"], errors="coerce").between(1, 31)]
    df["dia"] = df["dia"].astype(int)
    
    df["fecha"] = pd.to_datetime(
        df["dia"].apply(lambda d: f"{anio}-{mes:02d}-{d:02d}"),
        errors="coerce"
    )
    
    # Solo unpivot columnas que tienen formato planta__metrica
    cols_metricas = [c for c in df.columns if "__" in str(c)]
    
    df_long = df.melt(
        id_vars=["fecha", "dia"],
        value_vars=cols_metricas,
        var_name="planta_metrica",
        value_name="valor"
    )
    
    # Split seguro: n=1 garantiza máximo 2 partes
    split_df = df_long["planta_metrica"].str.split("__", n=1, expand=True)
    df_long["planta"] = split_df[0].str.strip()
    df_long["metrica"] = split_df[1].str.strip()
    df_long = df_long.drop(columns=["planta_metrica"])
    
    df_long["_source_file"] = nombre
    df_long["_ingested_at"] = INGESTED_AT
    df_long["_batch_id"] = BATCH_ID
    df_long["anio"] = anio
    df_long["mes"] = mes
    
    todos_df.append(df_long)
    print(f"  ✓ {nombre}: {len(df_long)} filas")

df_final = pd.concat(todos_df, ignore_index=True)
print(f"\nTotal filas: {len(df_final)}")
print(f"Plantas únicas: {sorted(df_final['planta'].unique())}")
print(f"Métricas únicas: {sorted(df_final['metrica'].unique())}")

df_final.head(10)

StatementMeta(, 08e49674-7cfa-41e4-a105-cc98de505606, 8, Finished, Available, Finished, False)

  ✓ 01 RESUMEN ENERO 2021.xlsx: 651 filas
  ✓ 02 RESUMEN FEBRERO 2021.xlsx: 588 filas
  ✓ 03 RESUMEN MARZO 2021.xlsx: 651 filas
  ✓ 04 RESUMEN ABRIL 2021.xlsx: 630 filas
  ✓ 05 RESUMEN MAYO 2021.xlsx: 651 filas
  ✓ 06 RESUMEN JUNIO 2021.xlsx: 630 filas
  ✓ 07 RESUMEN JULIO 2021.xlsx: 651 filas
  ✓ 08  RESUMEN AGOSTO 2021.xlsx: 651 filas
  ✓ 09 RESUMEN SEPTIEMBRE 2021.xlsx: 630 filas
  ✓ 10 RESUMEN OCTUBRE 2021.xlsx: 651 filas
  ✓ 11 RESUMEN NOVIEMBRE 2021.xlsx: 660 filas
  ✓ 12 RESUMEN DICIEMBRE 2021.xlsx: 651 filas

Total filas: 7695
Plantas únicas: ['ALAO', 'C.NIZAG', 'ENERGIA ENTREGADA AL MEM (kWh)', 'ENERGIA RECIBIDA DEL MEM (kWH)', 'RIO BLANCO', 'S.N.I.', 'TOTAL EERSA']
Métricas únicas: ['C.Int kWh', 'E.Bruta kWh', 'E.Neta Kwh', 'E.Neta kWh', 'KW', 'KW-H', 'KWh', 'KWh.1', 'KWh.2', 'KWh.3', 'Unnamed: 21_level_1', 'Unnamed: 22_level_1', 'Unnamed: 23_level_1']


,fecha,dia,valor,planta,metrica,_source_file,_ingested_at,_batch_id,anio,mes
0,2021-01-01,1,10044.586,ALAO,KW,01 RESUMEN ENERO 2021.xlsx,2026-04-10 13:35:55.268239,6d73c592-e4fa-4683-9bca-87dfa9ed57b6,2021,1
1,2021-01-02,2,10059.704,ALAO,KW,01 RESUMEN ENERO 2021.xlsx,2026-04-10 13:35:55.268239,6d73c592-e4fa-4683-9bca-87dfa9ed57b6,2021,1
2,2021-01-03,3,10074.495,ALAO,KW,01 RESUMEN ENERO 2021.xlsx,2026-04-10 13:35:55.268239,6d73c592-e4fa-4683-9bca-87dfa9ed57b6,2021,1
3,2021-01-04,4,9956.266,ALAO,KW,01 RESUMEN ENERO 2021.xlsx,2026-04-10 13:35:55.268239,6d73c592-e4fa-4683-9bca-87dfa9ed57b6,2021,1
4,2021-01-05,5,10053.612,ALAO,KW,01 RESUMEN ENERO 2021.xlsx,2026-04-10 13:35:55.268239,6d73c592-e4fa-4683-9bca-87dfa9ed57b6,2021,1
5,2021-01-06,6,8981.613,ALAO,KW,01 RESUMEN ENERO 2021.xlsx,2026-04-10 13:35:55.268239,6d73c592-e4fa-4683-9bca-87dfa9ed57b6,2021,1
6,2021-01-07,7,10179.354,ALAO,KW,01 RESUMEN ENERO 2021.xlsx,2026-04-10 13:35:55.268239,6d73c592-e4fa-4683-9bca-87dfa9ed57b6,2021,1
7,2021-01-08,8,10174.46,ALAO,KW,01 RESUMEN ENERO 2021.xlsx,2026-04-10 13:35:55.268239,6d73c592-e4fa-4683-9bca-87dfa9ed57b6,2021,1
8,2021-01-09,9,10178.28,ALAO,KW,01 RESUMEN ENERO 2021.xlsx,2026-04-10 13:35:55.268239,6d73c592-e4fa-4683-9bca-87dfa9ed57b6,2021,1
9,2021-01-10,10,10189.34,ALAO,KW,01 RESUMEN ENERO 2021.xlsx,2026-04-10 13:35:55.268239,6d73c592-e4fa-4683-9bca-87dfa9ed57b6,2021,1


In [7]:
# ============================================================
# Bronze: escribir tabla Delta en lh_bronze_eersa
# ============================================================

# Convertir pandas → Spark DataFrame
df_spark = spark.createDataFrame(df_final)

# Escribir como tabla Delta en el schema dbo (default del Lakehouse)
nombre_tabla = "bronze_generacion_diaria"

df_spark.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(nombre_tabla)

print(f"✓ Tabla escrita: {nombre_tabla}")
print(f"  Filas: {df_spark.count()}")
print(f"  Columnas: {len(df_spark.columns)}")

# Verificar leyendo de vuelta
spark.sql(f"SELECT COUNT(*) as total, COUNT(DISTINCT planta) as plantas FROM {nombre_tabla}").show()

StatementMeta(, 08e49674-7cfa-41e4-a105-cc98de505606, 9, Finished, Available, Finished, False)

/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Could not convert 'DM' with type str: tried to convert to double
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.


✓ Tabla escrita: bronze_generacion_diaria
  Filas: 7695
  Columnas: 10
+-----+-------+
|total|plantas|
+-----+-------+
| 7695|      7|
+-----+-------+

